# Gated Evidence Fusion Network

## The problem with Bayesian logLR fusion

The 3-channel Bayesian model (AUC 0.841) combines evidence by **summing logLRs**:
```
total_logLR = logLR_esim + logLR_gap + logLR_rt
```
This assumes channels are **independent** and combine **linearly** in log-space.

When we tried 9 channels (adding adduct features), AUC dropped — the independence
assumption broke down and correlated channels double-counted evidence.

## What gated fusion does differently

A gated network learns **when to trust each channel**, conditioned on all other channels:

```
gate_i = sigmoid(W_i · [all_features])     ← context-dependent reliability weight
gated_i = feature_i × gate_i               ← modulated evidence
output = MLP(concat(gated_1, ..., gated_k)) → P(correct)
```

Key interactions this can learn that Bayesian logLR cannot:
- "Trust RT less when compound is N-acetyl" (class-specific RT unreliability)
- "High esim + bad RT = suspicious" (non-additive interaction)
- "ISF adduct with no ok confirmation + high esim = likely fragment" (conditional evidence)
- "When n_candidates is large, sim_gap matters more" (context-dependent weighting)

## Models compared

| Model | Features | Fusion | Interactions |
|-------|----------|--------|-------------|
| Bayesian 3ch | 3 | sum of logLRs | none (independence) |
| GBM | 12 | gradient boosted trees | learned (axis-aligned splits) |
| Gated MLP | 12 | learned gates + MLP | learned (smooth, differentiable) |

All evaluated with **identical 5-fold GroupKFold by IK14** for fair comparison.

In [ ]:
import sys, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# ── Load data — same as bayesian_3channel_explorer ──
ft = pd.read_csv('../data/feature_table_v2.csv')
top1 = (ft.sort_values('entropy_similarity', ascending=False)
          .groupby('wiki_id').first().reset_index())

labels = top1['hit_label'].values

# ── Feature sets ──

# The 3 channels used by the Bayesian model
BAYES_COLS = ['entropy_similarity', 'sim_gap', 'signed_delta_rt']

# All 12 features — the full evidence set
ALL_COLS = [
    # Continuous evidence channels
    'entropy_similarity',   # MS2 spectral match quality
    'sim_gap',              # margin over runner-up candidate
    'signed_delta_rt',      # RT prediction error (seconds, signed)
    'delta_mda',            # mass accuracy (mDa)
    'spectral_entropy',     # query spectrum complexity

    # Discrete/binary evidence
    'hit_is_isf',           # candidate adduct is in-source fragment
    'hit_isf_no_ok',        # ISF with no molecular ion confirmation (high risk)
    'compound_has_ok_adduct', # compound has [M-H]- or [M+H]+ somewhere
    'n_candidate_adducts',  # how many adducts seen for this compound
    'n_candidates',         # total candidates for this spectrum

    # Context
    'hit_is_dubious',       # dubious adduct per Oliver's taxonomy
    'polarity',             # 0=neg, 1=pos
]

# Fill missing values with median (same strategy for all models)
medians = top1[ALL_COLS].median()
X_all = top1[ALL_COLS].fillna(medians).values.astype(np.float32)

# Build CV groups
groups = top1['anno_ik14'].fillna('').values.copy()
for i in range(len(groups)):
    if groups[i] == '':
        groups[i] = f'__no_ik14_{i}'

print(f'Spectra: {len(top1):,}  |  TP: {labels.sum():,}  |  FP: {(labels==0).sum():,}')
print(f'Features: {len(ALL_COLS)}')
print(f'Feature names: {ALL_COLS}')

## Step 1 — Model definitions

Three models, all receiving the same 12 features:

### GBM (tree-based baseline)
GradientBoostingClassifier — learns axis-aligned splits. Our strongest non-neural baseline.

### Plain MLP
Standard feedforward network. Tests whether a neural net helps without the gating mechanism.
This isolates whether the gain (if any) comes from "neural" or from "gating."

### Gated Fusion MLP
Each feature gets a **learned reliability gate** conditioned on all features.
The gate can suppress unreliable evidence and amplify informative evidence,
depending on the context of other features.

In [ ]:
class PlainMLP(nn.Module):
    """Standard feedforward network. No gating — just learns a function of all features."""

    def __init__(self, n_features, hidden=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class GatedFusionMLP(nn.Module):
    """
    Gated evidence fusion network.

    Architecture:
        1. Gate network: sees ALL features, outputs per-feature reliability weights
           gate_i = sigmoid(W_gate @ x + b_gate)[i]

        2. Gated features: element-wise product
           gated = x * gate

        3. Fusion MLP: processes gated features + raw features (residual)
           output = MLP(concat(gated, x)) → logit

    The gate learns context-dependent feature weighting:
    - When RT is unreliable (N-acetyl), gate_rt → 0
    - When esim is high but gap is 0 (isomers), gate_rt → 1 (upweight RT)
    - When adduct is ISF with no confirmation, gate_esim → lower
    """

    def __init__(self, n_features, hidden=64, dropout=0.3):
        super().__init__()
        self.n_features = n_features

        # Gate network: all features → per-feature gate values
        self.gate_net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_features),
            nn.Sigmoid(),  # gates are in [0, 1]
        )

        # Fusion MLP: gated features + raw features (residual connection)
        fusion_input = n_features * 2  # gated + raw
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        gates = self.gate_net(x)         # [batch, n_features] — per-feature gates
        gated = x * gates                # element-wise gating
        fused = torch.cat([gated, x], dim=-1)  # residual: model sees both gated and raw
        return self.fusion(fused).squeeze(-1)

    def get_gates(self, x):
        """Return gate values for interpretability."""
        with torch.no_grad():
            return self.gate_net(x).numpy()


def train_neural(model, X_train, y_train, X_val, y_val,
                 lr=1e-3, weight_decay=1e-4, epochs=200, batch_size=256, patience=20):
    """Train a PyTorch model with early stopping on validation AUC."""

    # Class weights to handle imbalance (69% TP, 31% FP)
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=7, factor=0.5)

    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32)
    X_v = torch.tensor(X_val, dtype=torch.float32)
    y_v = torch.tensor(y_val, dtype=torch.float32)

    dataset = TensorDataset(X_t, y_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_auc = 0
    best_state = None
    wait = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(X_v).numpy()
            val_probs = 1.0 / (1.0 + np.exp(-val_logits))
            val_auc = roc_auc_score(y_val, val_probs)

        scheduler.step(-val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return best_auc


print('Models defined:')
print(f'  PlainMLP:       {sum(p.numel() for p in PlainMLP(12).parameters()):,} params')
print(f'  GatedFusionMLP: {sum(p.numel() for p in GatedFusionMLP(12).parameters()):,} params')

## Step 2 — Head-to-head evaluation

Same 5-fold GroupKFold, same features, same labels. The only difference is the fusion method.

For neural models, we do a nested split: within each train fold, hold out 20% as a
validation set for early stopping. This prevents overfitting without touching the test fold.

In [ ]:
gkf = GroupKFold(n_splits=5)

# Storage for out-of-fold predictions
oof = {
    'gbm':   np.full(len(labels), np.nan),
    'mlp':   np.full(len(labels), np.nan),
    'gated': np.full(len(labels), np.nan),
}
fold_aucs = {k: [] for k in oof}

# Also store per-fold gate values for interpretability
all_gates = []  # list of (test_idx, gate_matrix) tuples

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_all, labels, groups)):
    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = labels[train_idx], labels[test_idx]

    # ── Standardize (fit on train only) ──
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr).astype(np.float32)
    X_te_s = scaler.transform(X_te).astype(np.float32)

    # ── Nested val split for early stopping (last 20% of train) ──
    n_val = int(len(X_tr_s) * 0.2)
    X_nn_tr, X_nn_val = X_tr_s[:-n_val], X_tr_s[-n_val:]
    y_nn_tr, y_nn_val = y_tr[:-n_val], y_tr[-n_val:]

    # ── 1. GBM ──
    gbm = GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    gbm.fit(X_tr_s, y_tr)
    oof['gbm'][test_idx] = gbm.predict_proba(X_te_s)[:, 1]

    # ── 2. Plain MLP ──
    torch.manual_seed(42 + fold)
    mlp = PlainMLP(len(ALL_COLS), hidden=64, dropout=0.3)
    train_neural(mlp, X_nn_tr, y_nn_tr, X_nn_val, y_nn_val,
                 lr=1e-3, weight_decay=1e-4, epochs=200, patience=20)
    mlp.eval()
    with torch.no_grad():
        logits = mlp(torch.tensor(X_te_s)).numpy()
        oof['mlp'][test_idx] = 1.0 / (1.0 + np.exp(-logits))

    # ── 3. Gated Fusion MLP ──
    torch.manual_seed(42 + fold)
    gated = GatedFusionMLP(len(ALL_COLS), hidden=64, dropout=0.3)
    train_neural(gated, X_nn_tr, y_nn_tr, X_nn_val, y_nn_val,
                 lr=1e-3, weight_decay=1e-4, epochs=200, patience=20)
    gated.eval()
    with torch.no_grad():
        logits = gated(torch.tensor(X_te_s)).numpy()
        oof['gated'][test_idx] = 1.0 / (1.0 + np.exp(-logits))

    # Capture gate values for this fold's test set
    gate_vals = gated.get_gates(torch.tensor(X_te_s))
    all_gates.append((test_idx, gate_vals))

    # Per-fold AUCs
    for name in oof:
        auc = roc_auc_score(y_te, oof[name][test_idx])
        fold_aucs[name].append(auc)

    print(f'Fold {fold}: GBM={fold_aucs["gbm"][-1]:.4f}  '
          f'MLP={fold_aucs["mlp"][-1]:.4f}  '
          f'Gated={fold_aucs["gated"][-1]:.4f}  '
          f'(n={len(test_idx)}, TP={y_te.sum()})')

# ── Overall AUCs ──
print(f'\n{"="*60}')
print(f'{"Model":<20}  {"OOF AUC":>8}  {"Fold min":>9}  {"Fold max":>9}')
print(f'{"-"*60}')
for name in oof:
    valid = ~np.isnan(oof[name])
    auc = roc_auc_score(labels[valid], oof[name][valid])
    print(f'{name:<20}  {auc:>8.4f}  {min(fold_aucs[name]):>9.4f}  {max(fold_aucs[name]):>9.4f}')
print(f'{"Bayesian 3ch":<20}  {"0.8414":>8}  {"(from explorer notebook)"}')
print(f'{"="*60}')

## Step 3 — ROC and FDR comparison

The real question isn't just AUC — it's whether ML fusion improves FDR at the thresholds
we actually care about (conf >= 0.9 is our primary operating point).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

colors = {'gbm': 'tab:green', 'mlp': 'tab:blue', 'gated': 'tab:red'}
labels_display = {'gbm': 'GBM (12 feat)', 'mlp': 'Plain MLP (12 feat)', 'gated': 'Gated MLP (12 feat)'}

# ── Panel 1: ROC curves ──
ax = axes[0]
for name in oof:
    valid = ~np.isnan(oof[name])
    auc = roc_auc_score(labels[valid], oof[name][valid])
    fpr, tpr, _ = roc_curve(labels[valid], oof[name][valid])
    ax.plot(fpr, tpr, color=colors[name], lw=2,
            label=f'{labels_display[name]} (AUC={auc:.3f})')

# Add Bayesian reference
ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
# Annotate Bayesian
ax.text(0.55, 0.35, 'Bayesian 3ch\nAUC=0.841', fontsize=9, color='gray',
        style='italic', transform=ax.transAxes)

# ── Panel 2: FDR vs threshold ──
ax = axes[1]
thresholds = np.linspace(0.01, 0.99, 200)

for name in oof:
    valid = ~np.isnan(oof[name])
    lab_v = labels[valid]
    pred_v = oof[name][valid]
    fdr = []
    for t in thresholds:
        called = pred_v >= t
        n = called.sum()
        fdr.append(((lab_v == 0) & called).sum() / n if n > 0 else 0)
    ax.plot(thresholds, fdr, color=colors[name], lw=2, label=labels_display[name])

ax.axhline(0.05, color='gray', linestyle=':', lw=1, label='5% FDR')
ax.axhline(0.10, color='gray', linestyle='--', lw=1, label='10% FDR')
ax.set_xlabel('Confidence threshold')
ax.set_ylabel('FDR')
ax.set_title('FDR vs threshold')
ax.set_xlim(0.3, 1.0)
ax.set_ylim(0, 0.3)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# ── Panel 3: FDR comparison table ──
ax = axes[2]
ax.axis('off')

rows = []
for name in ['gbm', 'mlp', 'gated']:
    valid = ~np.isnan(oof[name])
    lab_v = labels[valid]
    pred_v = oof[name][valid]
    auc = roc_auc_score(lab_v, pred_v)
    row = [labels_display[name], f'{auc:.3f}']
    for t in [0.9, 0.8, 0.7]:
        called = pred_v >= t
        n = called.sum()
        n_fp = ((lab_v == 0) & called).sum()
        fdr = n_fp / n if n > 0 else 0
        row.append(f'{fdr:.1%} ({n:,})')
    rows.append(row)

# Add Bayesian reference row
rows.append(['Bayesian 3ch', '0.841', '4.9% (1,047)', '6.6% (1,920)', '10.9% (3,038)'])

table = ax.table(
    cellText=rows,
    colLabels=['Model', 'AUC', 'FDR@0.9 (n)', 'FDR@0.8 (n)', 'FDR@0.7 (n)'],
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
ax.set_title('Head-to-head comparison', fontsize=12, pad=20)

plt.tight_layout()
plt.show()

## Step 4 — Gate interpretability

The main advantage of gated fusion over a black-box MLP: we can **inspect the gates**.

For each spectrum, the gate network produces 12 values in [0, 1] — one per feature.
A gate near 1 means "trust this feature"; near 0 means "suppress it."

Questions to answer:
1. Which features get gated down on average? (globally unreliable channels)
2. Do gates differ between TP and FP? (the network learned class-conditional weighting)
3. Are there spectrum subgroups with distinctive gate patterns? (e.g., ISF spectra)

In [ ]:
# Reconstruct full gate matrix from all folds
gate_matrix = np.zeros((len(labels), len(ALL_COLS)))
for test_idx, gates in all_gates:
    gate_matrix[test_idx] = gates

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── Panel 1: mean gate values per feature ──
ax = axes[0, 0]
mean_gates = gate_matrix.mean(axis=0)
order = np.argsort(mean_gates)
ax.barh(range(len(ALL_COLS)), mean_gates[order], color='steelblue')
ax.set_yticks(range(len(ALL_COLS)))
ax.set_yticklabels([ALL_COLS[i] for i in order])
ax.set_xlabel('Mean gate value (0=suppressed, 1=trusted)')
ax.set_title('Average gate activation per feature')
ax.axvline(0.5, color='gray', linestyle=':', lw=1)

# ── Panel 2: gate values by class (TP vs FP) ──
ax = axes[0, 1]
tp_gates = gate_matrix[labels == 1].mean(axis=0)
fp_gates = gate_matrix[labels == 0].mean(axis=0)

x = np.arange(len(ALL_COLS))
w = 0.35
ax.barh(x - w/2, tp_gates[order], w, color='steelblue', label='TP')
ax.barh(x + w/2, fp_gates[order], w, color='salmon', label='FP')
ax.set_yticks(x)
ax.set_yticklabels([ALL_COLS[i] for i in order])
ax.set_xlabel('Mean gate value')
ax.set_title('Gate activation: TP vs FP')
ax.legend()

# ── Panel 3: gate value distributions for top 4 most variable features ──
ax = axes[1, 0]
gate_std = gate_matrix.std(axis=0)
top4 = np.argsort(gate_std)[-4:][::-1]

for i, fi in enumerate(top4):
    tp_vals = gate_matrix[labels == 1, fi]
    fp_vals = gate_matrix[labels == 0, fi]
    positions = [i * 3, i * 3 + 1]
    bp = ax.boxplot([tp_vals, fp_vals], positions=positions, widths=0.6,
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor('salmon')
    bp['boxes'][1].set_alpha(0.6)

ax.set_xticks([i * 3 + 0.5 for i in range(4)])
ax.set_xticklabels([ALL_COLS[fi] for fi in top4], rotation=20, ha='right')
ax.set_ylabel('Gate value')
ax.set_title('Most variable gates (blue=TP, red=FP)')
ax.set_ylim(-0.05, 1.05)

# ── Panel 4: gate heatmap for ISF vs non-ISF spectra ──
ax = axes[1, 1]
isf_mask = top1['hit_is_isf'].values == 1
isf_gates = gate_matrix[isf_mask].mean(axis=0)
non_isf_gates = gate_matrix[~isf_mask].mean(axis=0)
diff = isf_gates - non_isf_gates

ax.barh(range(len(ALL_COLS)), diff[order], color=['salmon' if d < 0 else 'steelblue' for d in diff[order]])
ax.set_yticks(range(len(ALL_COLS)))
ax.set_yticklabels([ALL_COLS[i] for i in order])
ax.set_xlabel('Gate difference (ISF - non-ISF)')
ax.set_title(f'Gate shift for ISF spectra (n={isf_mask.sum():,}) vs non-ISF (n={(~isf_mask).sum():,})')
ax.axvline(0, color='black', linestyle=':', lw=0.8)

plt.tight_layout()
plt.show()

# Print summary
print('Feature gate summary (mean ± std):')
print(f'{"Feature":<25}  {"Mean":>6}  {"Std":>6}  {"TP mean":>8}  {"FP mean":>8}  {"Delta":>6}')
print('-' * 70)
for i in order[::-1]:
    print(f'{ALL_COLS[i]:<25}  {mean_gates[i]:>6.3f}  {gate_std[i]:>6.3f}  '
          f'{tp_gates[i]:>8.3f}  {fp_gates[i]:>8.3f}  {tp_gates[i]-fp_gates[i]:>+6.3f}')

## Step 5 — GBM feature importance comparison

Compare what the GBM learned (feature importance via split gain) with what the
gated network learned (gate activation patterns). If they agree, the signal is real.
If they disagree, one model is finding something the other missed.

In [ ]:
# Refit GBM on all data for feature importance (this isn't for evaluation, just for inspection)
scaler_full = StandardScaler()
X_full_s = scaler_full.fit_transform(X_all)
gbm_full = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gbm_full.fit(X_full_s, labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel 1: GBM feature importance ──
ax = axes[0]
imp = gbm_full.feature_importances_
order_imp = np.argsort(imp)
ax.barh(range(len(ALL_COLS)), imp[order_imp], color='tab:green')
ax.set_yticks(range(len(ALL_COLS)))
ax.set_yticklabels([ALL_COLS[i] for i in order_imp])
ax.set_xlabel('GBM feature importance (split gain)')
ax.set_title('GBM: which features does it split on?')

# ── Panel 2: gate activation vs GBM importance (scatter) ──
ax = axes[1]
gate_variability = gate_matrix.std(axis=0)  # high std = gate actively modulates this feature

ax.scatter(imp, gate_variability, s=80, color='darkorange', zorder=3)
for i, col in enumerate(ALL_COLS):
    ax.annotate(col, (imp[i], gate_variability[i]), fontsize=8,
                xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('GBM importance')
ax.set_ylabel('Gate variability (std across spectra)')
ax.set_title('GBM importance vs gate variability\n(top-right = both models find it important)')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## Step 6 — Error analysis: where does gated fusion disagree with Bayesian?

The most informative cases are where the two models **disagree strongly**:
- Bayesian confident + Gated uncertain → Bayesian is overconfident (interaction effect?)
- Bayesian uncertain + Gated confident → Gated found signal in extra features

Looking at these cases reveals what the gating mechanism actually learned.

In [ ]:
# Reproduce Bayesian OOF scores for comparison
sys.path.insert(0, '.')
from bayesian_score_v2 import ChannelSpec, fit_channel, logit_upper_half

BAYES_CHANNELS = [
    ChannelSpec('entropy_sim', 'entropy_similarity', 'continuous',
                higher_means_tp=True, tp_family='normal', fp_family='normal',
                transform=logit_upper_half),
    ChannelSpec('sim_gap', 'sim_gap', 'continuous',
                higher_means_tp=True, tp_family='normal', fp_family='normal'),
    ChannelSpec('signed_delta_rt', 'signed_delta_rt', 'continuous',
                higher_means_tp=True, tp_family='student_t', fp_family='student_t'),
]

top1_bayes = top1.copy()
top1_bayes['signed_delta_rt'] = top1_bayes['signed_delta_rt'].fillna(0)
prior_tp = labels.mean()
prior_log_odds = np.log(prior_tp / (1 - prior_tp))

oof_bayes = np.full(len(labels), np.nan)
for train_idx, test_idx in gkf.split(top1_bayes, labels, groups):
    fitted = []
    for spec in BAYES_CHANNELS:
        fc = fit_channel(spec, top1_bayes.iloc[train_idx][spec.feature_col].values, labels[train_idx])
        if fc is not None:
            fitted.append(fc)
    total_lr = np.zeros(len(test_idx))
    for fc in fitted:
        total_lr += fc.logLR(top1_bayes.iloc[test_idx][fc.spec.feature_col].values)
    oof_bayes[test_idx] = 1.0 / (1.0 + np.exp(-(prior_log_odds + total_lr)))

# ── Find disagreement cases ──
valid = ~np.isnan(oof['gated']) & ~np.isnan(oof_bayes)
diff = oof['gated'][valid] - oof_bayes[valid]

# Top disagreements
top1_valid = top1[valid].copy()
top1_valid['bayes_score'] = oof_bayes[valid]
top1_valid['gated_score'] = oof['gated'][valid]
top1_valid['score_diff'] = diff
top1_valid['label'] = labels[valid]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel 1: scatter of Bayesian vs Gated scores ──
ax = axes[0]
tp_mask = top1_valid['label'] == 1
ax.scatter(top1_valid.loc[tp_mask, 'bayes_score'],
           top1_valid.loc[tp_mask, 'gated_score'],
           s=5, alpha=0.3, color='steelblue', label='TP')
ax.scatter(top1_valid.loc[~tp_mask, 'bayes_score'],
           top1_valid.loc[~tp_mask, 'gated_score'],
           s=5, alpha=0.3, color='salmon', label='FP')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('Bayesian 3ch posterior')
ax.set_ylabel('Gated MLP posterior')
ax.set_title('Score agreement')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# ── Panel 2: where they disagree most ──
ax = axes[1]
ax.hist(diff[labels[valid] == 1], bins=50, alpha=0.5, color='steelblue',
        density=True, label='TP')
ax.hist(diff[labels[valid] == 0], bins=50, alpha=0.5, color='salmon',
        density=True, label='FP')
ax.axvline(0, color='black', linestyle=':', lw=0.8)
ax.set_xlabel('Gated score - Bayesian score')
ax.set_ylabel('Density')
ax.set_title('Score difference distribution')
ax.legend()

plt.tight_layout()
plt.show()

# Show biggest disagreement cases
print('=== Bayesian confident but Gated says NO (Bayesian overconfident?) ===')
overconf = top1_valid.nsmallest(5, 'score_diff')
for _, r in overconf.iterrows():
    correct = 'TP' if r['label'] == 1 else 'FP'
    print(f'  {correct}  bayes={r["bayes_score"]:.3f}  gated={r["gated_score"]:.3f}  '
          f'esim={r["entropy_similarity"]:.3f}  gap={r["sim_gap"]:.3f}  '
          f'rt={r["signed_delta_rt"]:.1f}  isf={int(r["hit_is_isf"])}  '
          f'name="{str(r["name"])[:30]}"')

print('\n=== Gated confident but Bayesian says NO (Gated found extra signal?) ===')
underconf = top1_valid.nlargest(5, 'score_diff')
for _, r in underconf.iterrows():
    correct = 'TP' if r['label'] == 1 else 'FP'
    print(f'  {correct}  bayes={r["bayes_score"]:.3f}  gated={r["gated_score"]:.3f}  '
          f'esim={r["entropy_similarity"]:.3f}  gap={r["sim_gap"]:.3f}  '
          f'rt={r["signed_delta_rt"]:.1f}  isf={int(r["hit_is_isf"])}  '
          f'name="{str(r["name"])[:30]}"')

## Step 7 — Bootstrap confidence intervals

Are the AUC differences real or just noise? Bootstrap on the OOF predictions
to get 95% CIs for each model and the pairwise differences.

In [ ]:
n_boot = 2000
boot_results = {name: [] for name in ['bayes', 'gbm', 'mlp', 'gated']}
boot_diffs = {f'gated-{other}': [] for other in ['bayes', 'gbm', 'mlp']}

all_preds = {
    'bayes': oof_bayes[valid],
    'gbm':   oof['gbm'][valid],
    'mlp':   oof['mlp'][valid],
    'gated': oof['gated'][valid],
}
lab_v = labels[valid]
n = len(lab_v)

for _ in range(n_boot):
    idx = np.random.choice(n, n, replace=True)
    if len(set(lab_v[idx])) < 2:
        continue
    for name, preds in all_preds.items():
        boot_results[name].append(roc_auc_score(lab_v[idx], preds[idx]))
    for other in ['bayes', 'gbm', 'mlp']:
        boot_diffs[f'gated-{other}'].append(
            boot_results['gated'][-1] - boot_results[other][-1]
        )

# Summary
print(f'{"Model":<12}  {"AUC":>8}  {"95% CI":>18}')
print('-' * 45)
for name in ['bayes', 'gbm', 'mlp', 'gated']:
    arr = np.array(boot_results[name])
    lo, hi = np.percentile(arr, [2.5, 97.5])
    auc_pt = roc_auc_score(lab_v, all_preds[name])
    print(f'{name:<12}  {auc_pt:>8.4f}  [{lo:.4f}, {hi:.4f}]')

print(f'\n{"Comparison":<15}  {"Delta":>8}  {"95% CI":>18}  {"Significant?"}')
print('-' * 60)
for key in boot_diffs:
    arr = np.array(boot_diffs[key])
    lo, hi = np.percentile(arr, [2.5, 97.5])
    sig = 'YES' if lo > 0 else ('YES (worse)' if hi < 0 else 'no')
    print(f'{key:<15}  {arr.mean():>+8.4f}  [{lo:+.4f}, {hi:+.4f}]  {sig}')

## Interpretation guide

### If Gated > GBM > Bayesian:
The gating mechanism captures feature interactions that both tree splits and
independent logLRs miss. The extra features + learned fusion = real improvement.
Next step: cross-attention transformer for even richer interactions.

### If GBM >= Gated > Bayesian:
The gain comes from extra features, not from gating specifically. GBM's tree splits
already capture the key interactions. The gating mechanism adds no value beyond what
GBM does for free. Next step: focus on feature engineering, not architecture.

### If all models are close (~0.84):
The 3 Bayesian channels already capture most of the signal. The extra 9 features and
learned interactions don't help. Next step: look for entirely new evidence types
(raw spectral peaks, structural embeddings) rather than better fusion of existing features.

### If Gated > MLP (same features):
The gating mechanism specifically helps — context-dependent feature weighting is valuable.
This motivates the cross-attention architecture (Step 2 from the AI directions discussion).